<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/notebooks/02_qfedavg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 2: q-FedAvg (Li et al. 2020)
Emaduddin Asdaq Syed Mohammed | CSC8639 MSc Data Science and AI

---
**This notebook runs q-FedAvg only, on the full frozen dataset.**

**Algorithm 2 — Li et al. 2020 (exact).** Each client computes its loss on
the current global model, `F_k(w_t)`, before local training; the server
then weights updates by `loss^q`, giving higher-loss (worse-performing)
hospitals more influence over the aggregated model — the opposite
weighting principle to FedAvg's size-based average.

**Parameters.** q = 0.5, following Li et al. The paper's recommended
`L ≈ 1/lr ≈ 10,000` stalled training entirely (the model failed to
update); `L = 1.0` was substituted pragmatically and is reported as a
documented implementation limitation (dissertation S3.2.2).

**Outcome (dissertation §4, Table 2 discussion).** This method did not
converge — global AUC 0.457, below the validity gate (>0.55) — and was
excluded from the fairness comparison. This notebook still runs to
completion and saves its metrics, since the failure itself is reported as
a finding (see dissertation §6, predicted-positive-rate ~81% evidence).

## Section 1 — Environment Setup

Installs Flower, imports all training/evaluation dependencies, mounts
Drive, and confirms GPU availability before any data handling begins.
Identical setup to the FedAvg notebook, so results are directly
comparable.

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.1
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr  : {fl.__version__}")
print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")
print(f"GPU   : {torch.cuda.is_available()}")

flwr  : 1.32.1
torch : 2.11.0+cu128
numpy : 2.0.2
GPU   : True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA   : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB
CUDA   : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

Re-downloads NIH ChestX-ray14 for this session.

In [ ]:
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/dissertation/kaggle.json',
            '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✓ Kaggle credentials loaded")

os.system('pip install -q kaggle')
os.system('kaggle datasets download -d nih-chest-xrays/data '
          '--path /content/nih_kaggle --unzip --quiet')

DATASET_PATH = '/content/nih_kaggle'
print(f"✓ Dataset path: {DATASET_PATH}")

✓ Kaggle credentials loaded
✓ Dataset path: /content/nih_kaggle


## Section 3 — Load Frozen Splits

Loads the same 15 frozen CSVs used by every method notebook, so q-FedAvg
trains and is evaluated on identical data to FedAvg and the other three
strategies.

In [ ]:
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']
NUM_CLIENTS = 5

train_clients = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_train.csv') for n in HOSPITAL_NAMES}
val_clients   = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_val.csv')   for n in HOSPITAL_NAMES}
test_clients  = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_test.csv')  for n in HOSPITAL_NAMES}
#Hospital_A_test.csv
for name in HOSPITAL_NAMES:
    print(f"{name}: {len(train_clients[name]):,} train / "
          f"{len(val_clients[name]):,} val / {len(test_clients[name]):,} test")

sample_path = train_clients['Hospital_A']['image_path'].iloc[0]
assert os.path.exists(sample_path), f"Path not found: {sample_path} — check Kaggle download completed"
print("✓ Image paths resolve correctly in this session")

Hospital_A: 52,332 train / 6,168 val / 3,005 test
Hospital_B: 9,074 train / 1,073 val / 508 test
Hospital_C: 29,813 train / 3,412 val / 1,711 test
Hospital_D: 1,276 train / 145 val / 74 test
Hospital_E: 2,997 train / 348 val / 168 test
✓ Image paths resolve correctly in this session


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

Sets DataLoader and cuDNN parameters, identical to the FedAvg notebook.

In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2


## Section 5 — Transforms, Dataset

Same preprocessing and `Dataset` wrapper as every other method notebook —
kept identical so aggregation strategy is the only variable under test.

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std")

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ Image size    : 224×224
✓ Normalisation : ImageNet mean/std
✓ ChestXrayDataset defined


## Section 6 — Model

Same ResNet-18 architecture as every method, so the aggregation rule
(Section 9) is the only difference from FedAvg.

In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]

✓ ResNet-18 — ImageNet pretrained
✓ Parameters  : 11,177,025
✓ Rounds      : 10


## Section 7 — Evaluation Function

Identical to the FedAvg notebook: computes AUC/FNR overall and per
subgroup, skipping subgroups with fewer than 10 samples or a single
class.

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")

✓ evaluate_client() defined
  Metrics : AUC + FNR (overall, per sex, per age group)


## Section 8 — Flower Base Client

Base per-hospital client, shared across method notebooks. q-FedAvg
subclasses this in Section 9 to add the pre-training loss computation
Li et al.'s algorithm requires.

In [ ]:
class HospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe
        self.val_dataframe = val_dataframe
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.val_dataframe), m

print("✓ HospitalClient defined")
print("  fit()      → trains on train_clients")
print("  evaluate() → validates on val_clients after each round")

✓ HospitalClient defined
  fit()      → trains on train_clients
  evaluate() → validates on val_clients after each round


## Section 9 — q-FedAvg (Li et al. 2020)

**Algorithm 2 — Li et al. 2020.** Each client sends `F_k(w_t)`, its loss
on the *current global model*, computed before local training. The
server then aggregates using:

`Δ_k = loss_k^q · (w_t − w̄_k)·L`,

`h_k = q · loss_k^(q−1) · ‖Δ_k‖² + L · loss_k^q`,

`w_new = w_t − Σ_k Δ_k / Σ_k h_k`

Higher pre-training loss (a struggling client) increases both `Δ_k` and
its effective weight — this is what gives worse-performing hospitals
more influence, in contrast to FedAvg's fixed size-based weighting.

In [ ]:
# Custom client: overrides fit() to compute F_k(w_t), the client's loss
# on the global model BEFORE any local training happens this round —
# required by Algorithm 2 above but not needed by plain FedAvg.
# The pre-loss pass uses num_workers=0 (no persistent_workers) because
# running two DataLoaders with persistent workers back-to-back in the
# same fit() call caused a Ray worker deadlock during testing.
class QFedAvgHospitalClient(HospitalClient):

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs    = int(config.get('epochs', 3))
        lr        = float(config.get('lr', 1e-4))
        criterion = nn.BCEWithLogitsLoss()

        pre_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False,
        )

        # F_k(w_t) — pre-training loss [Li et al. 2020 Algorithm 2]
        self.model.eval()
        loss_sum, n = 0.0, 0
        with torch.no_grad():
            for images, labels in pre_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                loss_sum += criterion(self.model(images), labels).item() * len(labels)
                n        += len(labels)
        loss_before = loss_sum / n

        # Local training proceeds exactly as in the FedAvg client —
        # the fairness weighting happens entirely server-side (aggregate_fit)
        train_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        self.model.train()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)
        total_loss, total_samples = 0.0, 0

        for _ in range(epochs):
            for images, labels in train_loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name,
             'loss_before': float(loss_before)}  # this is F_k(w_t), read by aggregate_fit below
        )

print("✓ QFedAvgHospitalClient defined")
print("  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]")

✓ QFedAvgHospitalClient defined
  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]


In [ ]:
# CELL — q-FedAvg client factory

def make_qfedavg_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid):
        name = HOSPITAL_NAMES[int(cid)]
        return QFedAvgHospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print("✓ q-FedAvg client factory defined")

✓ q-FedAvg client factory defined


In [ ]:
# q = 0.5 following Li et al. 2020.
# L should be ≈ 1/lr ≈ 10,000 per the paper, but that value stalled
# training entirely (no visible parameter update across rounds).
# L = 1.0 was substituted pragmatically; this is reported as a documented
# implementation limitation in the dissertation (§3.2.2), not treated as
# a silent fix — it likely explains this method's non-convergence
# (global AUC 0.457, below the 0.55 validity gate, §4).
Q_PARAM = 0.5
L_PARAM = 1.0

class QFedAvgStrategy(Strategy):
    # Custom Strategy (not a FedAvg subclass) because Li et al.'s
    # aggregation rule is structurally different from weighted averaging —
    # it needs each client's pre-training loss, which Flower's built-in
    # FedAvg has no mechanism to collect or use.

    def __init__(self, q=0.5, L=1.0, num_clients=5):
        super().__init__()
        self.q              = q
        self.L              = L
        self.num_clients    = num_clients
        self._global_params = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        return ndarrays_to_parameters(ndarrays)

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        ins     = FitIns(parameters, {'epochs': 3, 'lr': 1e-4})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        w_t       = self._global_params
        sum_Delta = None
        sum_h     = 0.0

        for _, fit_res in results:
            w_bar_k = parameters_to_ndarrays(fit_res.parameters)
            loss_k  = max(float(fit_res.metrics.get('loss_before', 1.0)), 1e-10)  # floored to avoid divide-by-zero in h_k below

            delta_w_k = [self.L * (w - wb) for w, wb in zip(w_t, w_bar_k)]
            Delta_k   = [(loss_k ** self.q) * dw for dw in delta_w_k]
            norm_sq   = float(sum(np.sum(dw ** 2) for dw in delta_w_k))
            h_k       = (self.q * (loss_k ** (self.q - 1)) * norm_sq
                         + self.L * (loss_k ** self.q))

            sum_Delta = Delta_k if sum_Delta is None else \
                        [sd + dk for sd, dk in zip(sum_Delta, Delta_k)]
            sum_h    += h_k

        if sum_h == 0.0 or sum_Delta is None:
            return ndarrays_to_parameters(w_t), {}

        w_new               = [w - sd / sum_h for w, sd in zip(w_t, sum_Delta)]
        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}

        print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
        for _, res in results:
            name = res.metrics.get('client_name', '?')
            auc  = res.metrics.get('auc', float('nan'))
            fnr  = res.metrics.get('fnr', float('nan'))
            print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
        aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
        valid_aucs = [a for a in aucs if not (a != a)]
        if valid_aucs:
            print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                  f"Variance : {float(np.var(valid_aucs)):.6f}")

        total_loss = sum(r.loss * r.num_examples for _, r in results)
        total_n    = sum(r.num_examples for _, r in results)
        return total_loss / total_n, {}

    def evaluate(self, server_round, parameters):
        return None  # centralised (server-side) evaluation not used — all evaluation is client-side via aggregate_evaluate above

print(f"✓ QFedAvgStrategy defined [Li et al. 2020]")
print(f"  q={Q_PARAM}  L={L_PARAM}")
print(f"  Note: L=1/lr=1e4 freezes model; L=1.0 adopted pragmatically")

✓ QFedAvgStrategy defined [Li et al. 2020]
  q=0.5  L=1.0
  Note: L=1/lr=1e4 freezes model; L=1.0 adopted pragmatically


### Run Training

Runs the 10-round q-FedAvg simulation using the custom strategy above.

In [ ]:
# CELL — Run q-FedAvg simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("q-FedAvg — Li et al. 2020")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"Split: 85/10/5 | Batch: {BATCH_SIZE}")
# NOTE: the line below prints "70/10/20" as a leftover from an earlier
# draft — it is a stray string only and is never used anywhere in the
# code. The actual split is 85/10/5, fixed by the frozen CSVs loaded in
# Section 3, and confirmed by the line above.
print(f"q={Q_PARAM}  L={L_PARAM} | Split: 70/10/20")
print("=" * 50)

t0               = time.time()
qfedavg_strategy = QFedAvgStrategy(q=Q_PARAM, L=L_PARAM, num_clients=NUM_CLIENTS)

qfedavg_history = fl.simulation.start_simulation(
    client_fn        = make_qfedavg_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = qfedavg_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*50}")
print(f"✓ q-FedAvg complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in qfedavg_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

q-FedAvg — Li et al. 2020
Rounds: 10 | Epochs/round: 3 | Clients: 5
Split: 85/10/5 | Batch: 512
q=0.5  L=1.0 | Split: 70/10/20


2026-07-18 13:09:07,374	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=23057) 
(ClientAppActor pid=23057)             This is a deprecated feature. It will be removed
(ClientAppActor pid=23057)             entirely in future versions of Flower.
(ClientAppActor pid=23057)         
(ClientAppActor pid=23057) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: Thi


── Round 1/10 Validation ──
  Hospital_A     AUC: 0.4651  FNR: 0.2252
  Hospital_B     AUC: 0.4966  FNR: 0.2131
  Hospital_C     AUC: 0.4634  FNR: 0.2258
  Hospital_D     AUC: 0.4756  FNR: 0.1635
  Hospital_E     AUC: 0.4542  FNR: 0.2500
  Mean AUC : 0.4710 | Variance : 0.000210


(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=23057) 
(ClientAppActor pid=23057)             This is a deprecated feature. It will be removed
(ClientAppActor pid=23057)             entirely in future versions of Flower.
(ClientAppActor pid=23057)         
(ClientAppActor pid=23057) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=23057) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=23057)   self.pid = os.fork()
(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 2/10 Validation ──
  Hospital_E     AUC: 0.4544  FNR: 0.2500
  Hospital_A     AUC: 0.4651  FNR: 0.2252
  Hospital_B     AUC: 0.4966  FNR: 0.2131
  Hospital_D     AUC: 0.4758  FNR: 0.1635
  Hospital_C     AUC: 0.4634  FNR: 0.2258
  Mean AUC : 0.4711 | Variance : 0.000209


...


── Round 8/10 Validation ──
  Hospital_B     AUC: 0.4969  FNR: 0.2140
  Hospital_E     AUC: 0.4541  FNR: 0.2500
  Hospital_C     AUC: 0.4634  FNR: 0.2258
  Hospital_D     AUC: 0.4756  FNR: 0.1635
  Hospital_A     AUC: 0.4651  FNR: 0.2255
  Mean AUC : 0.4710 | Variance : 0.000214


(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=23057) 
(ClientAppActor pid=23057)             This is a deprecated feature. It will be removed
(ClientAppActor pid=23057)             entirely in future versions of Flower.
(ClientAppActor pid=23057)         
(ClientAppActor pid=23057) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=23057) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=23057)   self.pid = os.fork()
(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 9/10 Validation ──
  Hospital_D     AUC: 0.4758  FNR: 0.1635
  Hospital_A     AUC: 0.4651  FNR: 0.2255
  Hospital_C     AUC: 0.4634  FNR: 0.2258
  Hospital_E     AUC: 0.4542  FNR: 0.2500
  Hospital_B     AUC: 0.4969  FNR: 0.2121
  Mean AUC : 0.4711 | Variance : 0.000214


(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=23057) 
(ClientAppActor pid=23057)             This is a deprecated feature. It will be removed
(ClientAppActor pid=23057)             entirely in future versions of Flower.
(ClientAppActor pid=23057)         
(ClientAppActor pid=23057) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=23057) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=23057)   self.pid = os.fork()
(ClientAppActor pid=23057) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 10/10 Validation ──
  Hospital_C     AUC: 0.4634  FNR: 0.2258
  Hospital_E     AUC: 0.4541  FNR: 0.2500
  Hospital_A     AUC: 0.4651  FNR: 0.2252
  Hospital_B     AUC: 0.4969  FNR: 0.2121
  Hospital_D     AUC: 0.4756  FNR: 0.1635
  Mean AUC : 0.4710 | Variance : 0.000214

✓ q-FedAvg complete in 670.1 minutes

Loss per round:
  Round  1 : 0.5326
  Round  2 : 0.5326
  Round  3 : 0.5326
  Round  4 : 0.5326
  Round  5 : 0.5326
  Round  6 : 0.5326
  Round  7 : 0.5326
  Round  8 : 0.5326
  Round  9 : 0.5326
  Round 10 : 0.5326


### Final Test Evaluation and Save

Evaluates the final aggregated model on each hospital's held-out test
set and saves metrics and weights to Drive.

In [ ]:
# CELL — Evaluate q-FedAvg per client

import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

qfedavg_metrics      = {}
qfedavg_final_params = qfedavg_strategy._global_params

for name, data in test_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in qfedavg_final_params]
    )))
    qfedavg_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = qfedavg_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_q = pd.DataFrame(rows).set_index('Client')
print("=== q-FedAvg Results — Li et al. 2020 ===\n")
print(df_q.to_string())

valid_aucs = [qfedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (qfedavg_metrics[n]['auc'] != qfedavg_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=5877) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=5877) is multi-threaded, use of fork() may lead to deadlocks in the child.
 

=== q-FedAvg Results — Li et al. 2020 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.4592  0.2192     52.98  0.4643  0.2230  0.4512  0.2139
Hospital_B     NaN  0.2146     78.54     NaN  0.2198     NaN  0.2085
Hospital_C  0.4444  0.2222     21.04  0.4971  0.1571  0.3858  0.2923
Hospital_D  0.4458  0.2308     59.46  0.4333  0.2121  0.4912  0.2632
Hospital_E  0.4659  0.2105     27.38  0.4433  0.4000  0.4802  0.0000

AUC Variance (valid hospitals only) : 0.000082
Valid hospitals : 4/5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL — Save q-FedAvg results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/qfedavg_full_metrics.json', 'w') as f:
    json.dump(qfedavg_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in qfedavg_final_params])),
    f'{SAVE_DIR}/qfedavg_full_model.pth'
)

print("✓ q-FedAvg metrics saved → qfedavg_full_metrics.json")
print("✓ q-FedAvg model saved  → qfedavg_full_model.pth")

✓ q-FedAvg metrics saved → qfedavg_full_metrics.json
✓ q-FedAvg model saved  → qfedavg_full_model.pth
